In [1]:
# Lab 13 - SQL en Python
#nombre : Juan Carlos Concha Hernandez
#RUT: 21.684.318-5 
#Fecha: 2026-06-17

In [2]:
# ── EJERCICIO 1: Crear Base de Datos y Tabla ──
import sqlite3
conn = sqlite3.connect("clima_lab.db")
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS registros (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        ciudad     TEXT NOT NULL,
        temp_c     REAL,
        humedad    INTEGER,
        precip_mm  REAL,
        viento_kmh REAL,
        fecha      TEXT
    )
""")

conn.commit()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='registros'")
if cursor.fetchone():
    print("Tabla 'registros' creada.")

cursor.execute("PRAGMA table_info(registros)")
print("Columnas:")
for col in cursor.fetchall():
    print(f"  {col[1]}  {col[2]}")
conn.close()

Tabla 'registros' creada.
Columnas:
  id  INTEGER
  ciudad  TEXT
  temp_c  REAL
  humedad  INTEGER
  precip_mm  REAL
  viento_kmh  REAL
  fecha  TEXT


In [3]:
# ── EJERCICIO 2: Inserción Segura con Parámetros Vinculados ──
conn = sqlite3.connect("clima_lab.db")
cursor = conn.cursor()
# Insertar un registro con execute()
cursor.execute("""
    INSERT INTO registros (ciudad, temp_c, humedad, precip_mm, viento_kmh, fecha)
    VALUES (?, ?, ?, ?, ?, ?)
""", ('Santiago', 18.5, 72, 1.2, 15.3, '2026-06-15'))
conn.commit()
print("ID asignado:", cursor.lastrowid)
# Insertar 4 registros con executemany()
datos = [
    ('Valparaíso', 15.2, 80, 3.5, 22.1, '2026-06-15'),
    ('Concepción', 12.8, 85, 8.0, 18.7, '2026-06-15'),
    ('Temuco',      9.3, 90, 12.5, 25.0, '2026-06-15'),
    ('La Serena',  17.8, 55, 0.0,  12.4, '2026-06-15'),
]
cursor.executemany("""
    INSERT INTO registros (ciudad, temp_c, humedad, precip_mm, viento_kmh, fecha)
    VALUES (?, ?, ?, ?, ?, ?)
""", datos)
conn.commit()
cursor.execute("SELECT COUNT(*) FROM registros")
print("Total de registros:", cursor.fetchone()[0])
conn.close()

ID asignado: 6
Total de registros: 9


In [4]:
# ── EJERCICIO 3: Consultas SELECT con Filtros y Agregaciones ──
conn = sqlite3.connect("clima_lab.db")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()
# Consulta 1 - Listado completo
cursor.execute("SELECT ciudad, temp_c, fecha FROM registros")
for row in cursor.fetchall():
    print(f"  {row['ciudad']:<15} {row['temp_c']}°C  {row['fecha']}")
# Consulta 2 - Ciudad más cálida
cursor.execute("SELECT ciudad, temp_c FROM registros ORDER BY temp_c DESC")
row = cursor.fetchone()
print(f"  {row['ciudad']} con {row['temp_c']}°C")
# Consulta 3 - Filtro por temperatura < 13°C
cursor.execute("SELECT ciudad, temp_c FROM registros WHERE temp_c < ?", (13.0,))
for row in cursor.fetchall():
    print(f"  {row['ciudad']}: {row['temp_c']}°C")
# Consulta 4 - Estadísticas por ciudad
cursor.execute("""
    SELECT ciudad,
           AVG(temp_c) AS prom_temp,
           MIN(humedad) AS hum_min,
           MAX(humedad) AS hum_max
    FROM registros
    GROUP BY ciudad
    ORDER BY prom_temp DESC
""")
for row in cursor.fetchall():
    print(f"  {row['ciudad']:<15} prom={row['prom_temp']:.1f}°C  hum min={row['hum_min']}  hum max={row['hum_max']}")
conn.close()

  Santiago        18.5°C  2026-06-15
  Valparaíso      16.0°C  2026-06-15
  Concepción      12.8°C  2026-06-15
  La Serena       17.8°C  2026-06-15
  Santiago        18.5°C  2026-06-15
  Valparaíso      15.2°C  2026-06-15
  Concepción      12.8°C  2026-06-15
  Temuco          9.3°C  2026-06-15
  La Serena       17.8°C  2026-06-15
  Santiago con 18.5°C
  Concepción: 12.8°C
  Concepción: 12.8°C
  Temuco: 9.3°C
  Santiago        prom=18.5°C  hum min=72  hum max=72
  La Serena       prom=17.8°C  hum min=55  hum max=55
  Valparaíso      prom=15.6°C  hum min=80  hum max=80
  Concepción      prom=12.8°C  hum min=85  hum max=85
  Temuco          prom=9.3°C  hum min=90  hum max=90


In [5]:
# ── EJERCICIO 4: UPDATE y DELETE con Validación ──
conn = sqlite3.connect("clima_lab.db")
cursor = conn.cursor()
# Operación 1 - UPDATE Valparaíso
cursor.execute("UPDATE registros SET temp_c = ? WHERE ciudad = ?", (16.0, 'Valparaíso'))
conn.commit()
print(f"Operación 1: {cursor.rowcount} fila(s) actualizada(s)")
cursor.execute("SELECT ciudad, temp_c FROM registros WHERE ciudad = ?", ('Valparaíso',))
row = cursor.fetchone()
print(f"  Verificación: {row[0]} → {row[1]}°C")
# Operación 2 - UPDATE masivo humedad
cursor.execute("UPDATE registros SET humedad = humedad + 5 WHERE temp_c < ?", (10.0,))
conn.commit()
print(f"\nOperación 2: {cursor.rowcount} fila(s) con humedad actualizada")
# Operación 3 - DELETE Temuco
cursor.execute("DELETE FROM registros WHERE ciudad = ?", ('Temuco',))
conn.commit()
print(f"\nOperación 3: {cursor.rowcount} fila(s) eliminada(s)")
cursor.execute("SELECT COUNT(*) FROM registros")
print(f"  Total registros restantes: {cursor.fetchone()[0]}")
conn.close()

Operación 1: 2 fila(s) actualizada(s)
  Verificación: Valparaíso → 16.0°C

Operación 2: 1 fila(s) con humedad actualizada

Operación 3: 1 fila(s) eliminada(s)
  Total registros restantes: 8


In [6]:
# ── EJERCICIO 5: Pipeline CSV → Limpieza → SQLite → Análisis SQL ──
import pandas as pd
# EXTRACT
df = pd.read_csv("S15_registros_climaticos.csv")
print("=== Info del dataset ===")
print(df.info())
print("\nNulos por columna:")
print(df.isnull().sum())
filas_originales = len(df)
# TRANSFORM
df['ciudad'] = df['ciudad'].str.title()
df['fecha'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce').dt.strftime('%Y-%m-%d')
df = df.drop_duplicates(subset=['ciudad', 'fecha'])
df = df[df['temp_c'].between(-5, 50)]
filas_limpias = len(df)
# LOAD
conn = sqlite3.connect("clima_lab.db")
df.to_sql('registros_csv', conn, if_exists='replace', index=False)
print("\n=== Carga completada ===")
# ANÁLISIS SQL
print("\n=== Ciudad con mayor temperatura promedio ===")
resultado = pd.read_sql_query("""
    SELECT ciudad, AVG(temp_c) AS prom_temp
    FROM registros_csv
    GROUP BY ciudad
    ORDER BY prom_temp DESC
    LIMIT 1
""", conn)
print(resultado.to_string(index=False))
print("\n=== Promedio de precipitación por ciudad (Ene-Mar 2026) ===")
resultado2 = pd.read_sql_query("""
    SELECT ciudad, AVG(precip_mm) AS prom_precip
    FROM registros_csv
    WHERE fecha BETWEEN '2026-01-01' AND '2026-03-31'
    GROUP BY ciudad
    ORDER BY prom_precip DESC
""", conn)
print(resultado2.to_string(index=False))
print("\n=== Top 5 fechas con mayor temperatura ===")
resultado3 = pd.read_sql_query("""
    SELECT fecha, ciudad, temp_c
    FROM registros_csv
    ORDER BY temp_c DESC
    LIMIT 5
""", conn)
print(resultado3.to_string(index=False))
# RESUMEN FINAL
print("\n=== Resumen Final ===")
print(f"  Filas originales : {filas_originales}")
print(f"  Filas eliminadas : {filas_originales - filas_limpias}")
print(f"  Filas cargadas   : {filas_limpias}")
conn.close()

=== Info del dataset ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ciudad      208 non-null    object 
 1   temp_c      194 non-null    float64
 2   humedad     198 non-null    float64
 3   precip_mm   208 non-null    float64
 4   viento_kmh  208 non-null    float64
 5   fecha       208 non-null    object 
dtypes: float64(4), object(2)
memory usage: 9.9+ KB
None

Nulos por columna:
ciudad         0
temp_c        14
humedad       10
precip_mm      0
viento_kmh     0
fecha          0
dtype: int64

=== Carga completada ===

=== Ciudad con mayor temperatura promedio ===
     ciudad  prom_temp
Antofagasta  20.673333

=== Promedio de precipitación por ciudad (Ene-Mar 2026) ===
      ciudad  prom_precip
    Santiago     7.125000
    Rancagua     3.620000
   La Serena     3.514286
      Temuco     3.433333
       Talca     3.400000
 Antofagasta   

C:\Users\jcche\AppData\Local\Temp\ipykernel_5004\2643807726.py:12: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['fecha'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce').dt.strftime('%Y-%m-%d')
